# 실험 01: Tool 정의와 바인딩 (bind_tools)

## 1. 실험 제목과 목표
- **제목**: LLM에게 손과 발 달아주기 기초
- **목표**: 일반적인 파이썬 함수를 LangChain의 `@tool` 데코레이터를 이용해 LLM이 이해할 수 있는 형태(스키마)로 변환하고, 모델에 연결(`bind_tools`)하여 LLM이 스스로 "아, 이 질문은 툴을 써야겠구나!"라고 판단하는 과정을 관찰합니다.

## 2. 실험 개요
1. **실험 1**: 단순 덧셈 함수를 만들고, 일반 LLM과 Tool이 바인딩된 LLM의 응답 차이 비교
2. **실패/주의 케이스**: Tool의 설명(docstring)이나 파라미터 힌트가 부실할 때 모델이 툴을 오작동시키는 현상 관찰

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 실험 1: Tool 정의 및 바인딩 (bind_tools)
아주 평범한 파이썬 함수 위에 `@tool`이라는 마법의 데코레이터를 붙입니다.

In [3]:
# 1. 파이썬 함수를 Tool로 정의
# 매우 중요: 함수 안의 따옴표 3개(docstring)와 변수 타입 힌트(int)가 곧 LLM을 위한 프롬프트가 됩니다!
@tool
def add_numbers(a: int, b: int) -> int:
    """두 개의 정수를 입력받아 덧셈 결과를 반환하는 계산기입니다."""
    return a + b

print("=== [1. 일반 모델의 응답] ===")
question = "2495 더하기 3812는 뭐야?"
normal_res = llm.invoke(question)
print("텍스트 응답:", normal_res.content)
# 겉보기엔 정답을 맞출 수 있지만, 이는 '계산'한 게 아니라 학습된 패턴으로 '찍은' 것입니다. 숫자가 커지면 틀립니다.

print("\n=== [2. Tool이 바인딩된 모델의 응답] ===")
# 2. 모델에게 "너 이 툴(손과 발)을 써도 돼"라고 알려줍니다.
llm_with_tools = llm.bind_tools([add_numbers])

tool_res = llm_with_tools.invoke(question)
print("텍스트 응답:", tool_res.content) # 텍스트는 비어 있습니다! 대신...
print("Tool Calls 요처:", tool_res.tool_calls)

print("\n-> 결과: 일반 모델은 말(텍스트)로 대답했지만, 툴을 가진 모델은 '아 이건 텍스트로 답할 게 아니라 add_numbers 함수를 a=2495, b=3812로 실행해 줘!'라는 구조화된 명령(tool_calls)을 내렸습니다.")

=== [1. 일반 모델의 응답] ===
텍스트 응답: 2495 더하기 3812는 6307입니다.

=== [2. Tool이 바인딩된 모델의 응답] ===
텍스트 응답: 
Tool Calls 요처: [{'name': 'add_numbers', 'args': {'a': 2495, 'b': 3812}, 'id': 'call_gTQq2P8DqwYjsipHsJL3NaDU', 'type': 'tool_call'}]

-> 결과: 일반 모델은 말(텍스트)로 대답했지만, 툴을 가진 모델은 '아 이건 텍스트로 답할 게 아니라 add_numbers 함수를 a=2495, b=3812로 실행해 줘!'라는 구조화된 명령(tool_calls)을 내렸습니다.


## 6. 실패/주의 케이스: 불친절한 Tool 정의
만약 개발자가 귀찮아서 docstring도 안 쓰고 타입 힌트도 대충 쓰면 어떻게 될까요?

In [ ]:
print("[불친절한 Tool 시뮬레이션]")

@tool
def calculate_something(x, y):
    # 설명도 없고 타입도 없습니다.
    return x * y

bad_llm = llm.bind_tools([calculate_something])

try:
    bad_res = bad_llm.invoke("물건 3개를 500원에 샀어. 총 얼마야?")
    print("Tool Calls:", bad_res.tool_calls)
    print("\n-> 주의: 똑똑한 최신 모델은 대충 알아듣고 인자를 넣어주기도 하지만, 과거 모델이나 복잡한 툴의 경우 인자를 딕셔너리로 넣거나 순서를 뒤집는 등 오작동(Hallucination)이 심하게 발생합니다.")
except Exception as e:
    print("🔥 에러 발생:", e)

print("   결론: Tool Calling에서 함수의 Docstring과 Type Hint는 '코드'가 아니라 LLM을 조종하는 '프롬프트' 그 자체입니다. 무조건 상세히 적어야 합니다.")

## 7. 결과 해석

1. **`@tool` 데코레이터**: 파이썬 함수를 OpenAI나 Anthropic이 요구하는 JSON Schema 규격(이 함수는 이름이 뭐고, 파라미터는 뭐고, 설명은 뭐다)으로 자동 변환해 주는 강력한 마법사입니다.
2. **`bind_tools`**: LLM 객체에 툴 스키마를 주입합니다.
3. **`tool_calls`**: 모델이 툴이 필요하다고 판단하면, `content`(텍스트)를 비워두고 `tool_calls` 리스트에 툴 이름과 파라미터(`args`)를 담아 반환합니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* LLM이 `tool_calls`를 반환한다는 건, "나 계산 좀 대신 해줘!"라고 **요청**만 한 상태임.
* 툴은 스스로 실행되지 않음! 교안 슬라이드에 적혀 있듯 **'애플리케이션 코드가 Tool을 실행'**해야 함.

### 다음 개선 방향
* LLM이 내뱉은 `tool_calls`의 인자값(`args`)을 받아서, 파이썬 코드 레벨에서 실제로 계산을 수행하고, 그 결과를 다시 LLM에게 돌려주어 최종 답변을 받아내는 전체 흐름을 다음 노트북에서 조립해야 함.